# Roadwork modeling

This notebook runs a baseline model on `data/derived/street_weather_lagged_model.csv`.


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from pathlib import Path

DERIVED_DIR = Path("data/derived")


In [2]:
df = pd.read_csv(DERIVED_DIR / "street_weather_lagged_model.csv")
summary = {
    "rows": len(df),
    "years": (int(df["year"].min()), int(df["year"].max())),
    "positive_rate": float(df["roadwork_done"].mean()),
    "feature_count": df.shape[1] - 3,
}
summary


{'rows': 276000,
 'years': (2003, 2025),
 'positive_rate': 0.06987318840579711,
 'feature_count': 32}

In [3]:
predictors = [c for c in df.columns if c not in ["normalized_street_name", "year", "roadwork_done"]]
X = df[predictors].replace([float("inf"), float("-inf")], pd.NA).fillna(0)
y = df["roadwork_done"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]


In [4]:
metrics = {
    "roc_auc": roc_auc_score(y_test, y_prob),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}
metrics

report = pd.DataFrame(classification_report(y_test, y_pred, digits=4, output_dict=True)).T
report


,precision,recall,f1-score,support
0,0.941715,0.791755,0.860249,77015.000000
1,0.111419,0.347623,0.168751,5785.000000
accuracy,0.760725,0.760725,0.760725,0.760725
macro avg,0.526567,0.569689,0.514500,82800.000000
weighted avg,0.883705,0.760725,0.811936,82800.000000


In [5]:
feature_importance = (
    pd.Series(clf.feature_importances_, index=predictors)
    .sort_values(ascending=False)
    .head(15)
    .rename("importance")
)
feature_importance


roadwork_factor_summer_lag1        0.368978
roadwork_factor_summer_lag2        0.309788
temperature_curwinter              0.020839
wind_speed_winter_lag1             0.020336
wind_speed_summer_lag1             0.019496
wind_speed_summer_lag2             0.016729
temperature_summer_lag2            0.014921
wind_speed_curwinter               0.014700
wind_speed_winter_lag2             0.014615
temperature_summer_lag1            0.014146
temperature_winter_lag1            0.013740
temperature_winter_lag2            0.013191
precipitation_summer_lag1          0.012946
precipitation_hours_winter_lag1    0.012941
rain_summer_lag1                   0.011172
Name: importance, dtype: float64

Expected result from the current table: prior summer roadwork features matter most, and weather variables matter less.
